# Retail Analytics ETL Pipeline — Simple Local Python Version

This notebook runs in ordinary Jupyter Notebook on your laptop. It does **not** require PySpark, Hadoop, Java, Azure, or `winutils.exe`.

Keep `sales_data.csv` and `footfall_data.csv` in the same Jupyter folder as these notebooks (your screenshot shows both files in the Jupyter root folder).

Run the notebooks in this order: **Bronze → Silver → Gold**.

## Gold Layer
Create hourly and daily business KPI files from the Silver CSV. Outputs are simple CSV files that can be opened in Excel or imported into Power BI.


In [ ]:
from pathlib import Path
import csv
from collections import defaultdict

BASE_DIR = Path.cwd()
output_root = BASE_DIR / "retail_analytics_output"
silver_file = output_root / "silver" / "hourly_store_performance.csv"

if not silver_file.exists():
    raise FileNotFoundError("Silver CSV not found. Run silver_notebook.ipynb first.")

with open(silver_file, newline="", encoding="utf-8-sig") as f:
    silver_rows = list(csv.DictReader(f))

print("Silver rows loaded:", len(silver_rows))


In [ ]:
# Prepare Gold hourly KPI output
gold_hourly_rows = []

for row in silver_rows:
    visitors = int(row["Footfall_Count"])
    transactions = int(row["Total_Transactions"])
    conversion = transactions / visitors * 100 if visitors > 0 else 0.0

    gold_hourly_rows.append({
        "Store_ID": row["Store_ID"],
        "Date": row["Date"],
        "Hour": int(row["Hour"]),
        "Total_Transactions": transactions,
        "Total_Revenue": round(float(row["Total_Revenue"]), 2),
        "Total_Footfall": visitors,
        "Conversion_Rate_Pct": round(conversion, 2)
    })

print("Gold hourly rows:", len(gold_hourly_rows))
for row in gold_hourly_rows[:5]:
    print(row)


In [ ]:
# Aggregate daily KPIs by store and date
daily_totals = defaultdict(lambda: {
    "Total_Transactions": 0,
    "Total_Revenue": 0.0,
    "Total_Footfall": 0
})

for row in gold_hourly_rows:
    key = (row["Store_ID"], row["Date"])
    daily_totals[key]["Total_Transactions"] += row["Total_Transactions"]
    daily_totals[key]["Total_Revenue"] += row["Total_Revenue"]
    daily_totals[key]["Total_Footfall"] += row["Total_Footfall"]

gold_daily_rows = []
for (store_id, date), values in sorted(daily_totals.items()):
    visitors = values["Total_Footfall"]
    transactions = values["Total_Transactions"]
    conversion = transactions / visitors * 100 if visitors > 0 else 0.0

    gold_daily_rows.append({
        "Store_ID": store_id,
        "Date": date,
        "Total_Transactions": transactions,
        "Total_Revenue": round(values["Total_Revenue"], 2),
        "Total_Footfall": visitors,
        "Conversion_Rate_Pct": round(conversion, 2)
    })

print("Gold daily rows:", len(gold_daily_rows))


In [ ]:
# Save Gold hourly and daily CSV outputs
gold_dir = output_root / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

hourly_fields = [
    "Store_ID", "Date", "Hour", "Total_Transactions",
    "Total_Revenue", "Total_Footfall", "Conversion_Rate_Pct"
]
daily_fields = [
    "Store_ID", "Date", "Total_Transactions",
    "Total_Revenue", "Total_Footfall", "Conversion_Rate_Pct"
]

hourly_file = gold_dir / "hourly_store_performance.csv"
daily_file = gold_dir / "daily_store_performance.csv"

with open(hourly_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=hourly_fields)
    writer.writeheader()
    writer.writerows(gold_hourly_rows)

with open(daily_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=daily_fields)
    writer.writeheader()
    writer.writerows(gold_daily_rows)

print("Gold hourly saved:", hourly_file)
print("Gold daily saved:", daily_file)


In [ ]:
# Store-level summary for Power BI
store_summary = defaultdict(lambda: {
    "Total_Transactions": 0,
    "Total_Revenue": 0.0,
    "Total_Footfall": 0
})

for row in gold_daily_rows:
    store_id = row["Store_ID"]
    store_summary[store_id]["Total_Transactions"] += row["Total_Transactions"]
    store_summary[store_id]["Total_Revenue"] += row["Total_Revenue"]
    store_summary[store_id]["Total_Footfall"] += row["Total_Footfall"]

print("STORE SUMMARY")
for store_id, values in sorted(store_summary.items()):
    visitors = values["Total_Footfall"]
    conversion = values["Total_Transactions"] / visitors * 100 if visitors else 0.0
    print({
        "Store_ID": store_id,
        "Total_Transactions": values["Total_Transactions"],
        "Total_Revenue": round(values["Total_Revenue"], 2),
        "Total_Footfall": visitors,
        "Conversion_Rate_Pct": round(conversion, 2)
    })

print("\nGold layer completed successfully.")


## Note
This simple local version calculates sales, footfall, revenue, transactions, and conversion rate. Category-wise return percentage is not available because the provided source CSV files do not contain product category or return records.
